# 03 · Embeddings and semantic search

**Level:** intermediate · **Time:** 40–50 minutes · **Prerequisite:** lists and basic NumPy.

**Learning outcome:** compute cosine similarity, inspect a document/query matrix, run real
local sentence embeddings, compare with TF-IDF, and evaluate paraphrase retrieval.
Claude is the answer generator; **Sentence Transformers provides embeddings locally**.
No embedding API key is needed. The first live run downloads the configured embedding model.

`RAG_OFFLINE=1` uses an explicitly labeled **lexical TF-IDF fallback**, not pretend semantic
vectors. It demonstrates mechanics without downloads; rerun live to evaluate semantic behavior.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the repository or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "rag_workshop").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the LangchainPractical project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rag_workshop import (
    Document, load_documents, chunk_documents, TfidfIndex, DenseIndex,
    bm25_search, reciprocal_rank_fusion, context_text, ask_claude,
    rag_answer, show_hits, settings, offline_mode,
)
import pandas as pd
from IPython.display import display, Markdown

print("Mode:", "OFFLINE: Claude calls skipped" if offline_mode() else "LIVE: Claude calls use API credits")
print("Claude model:", settings().model)


## 1. Understand vectors with a tiny example

A vector is a list of numbers. In this toy example, two coordinates represent how strongly
a sentence concerns "leave" and "API limits". Real embeddings learn hundreds of coordinates;
individual coordinates generally do not correspond to named human concepts.
Cosine similarity measures direction rather than vector length.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

toy_vectors = np.array([[3.0, 0.3], [2.0, 0.4], [0.2, 4.0]])
toy_labels = ["annual vacation", "paid time off", "API throughput"]
toy_query = np.array([1.0, 0.0])

def unit_rows(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.maximum(norms, 1e-12)

normalized_toy = unit_rows(toy_vectors)
normalized_query = toy_query / np.linalg.norm(toy_query)
toy_scores = normalized_toy @ normalized_query
display(pd.DataFrame({"text": toy_labels, "cosine": toy_scores,
                       "original_norm": np.linalg.norm(toy_vectors, axis=1)}))


## 2. Visualize direction versus magnitude

Vectors pointing in nearly the same direction have high cosine similarity even when
their lengths differ. After normalization, the dot product equals cosine similarity.
A zero vector needs explicit handling because it has no direction.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for vector, label in zip(toy_vectors, toy_labels):
    ax.quiver(0, 0, vector[0], vector[1], angles="xy", scale_units="xy", scale=1)
    ax.text(vector[0] + 0.06, vector[1], label)
ax.set(xlim=(-0.1, 4.5), ylim=(-0.1, 4.7), xlabel="Toy leave coordinate",
       ylabel="Toy API coordinate", title="Similar direction, different lengths")
ax.grid(alpha=0.2)
plt.show()
assert np.allclose(np.linalg.norm(normalized_toy, axis=1), 1.0)


## 3. Build two indexes over the exact same chunks

TF-IDF emphasizes words that are informative within this corpus. A pretrained sentence
model can represent relationships between paraphrases, but it can still miss details,
numbers, or domain terminology. We compare both methods with identical data.

`DenseIndex` uses the configured local Sentence Transformer in live mode. It does not
send documents to Anthropic. If the model cannot download, fix connectivity or explicitly
choose offline mode; do not silently label lexical results "semantic".


In [ ]:
documents = load_documents()
chunks = chunk_documents(documents, chunk_size=90, overlap=20)
lexical_index = TfidfIndex(chunks)
dense_index = DenseIndex(chunks)
print("Active backend:", dense_index.backend)
print("Embedding model requested:", settings().embedding_model)
print("Vector matrix shape [chunks, dimensions]:", dense_index.vectors.shape)
print("This run measures semantic embeddings:", not offline_mode())
if not offline_mode():
    # The actual local model API; the wrapper uses this for every document.
    example_vector = dense_index.embedder.encode(
        ["A short example sentence"], normalize_embeddings=True, convert_to_numpy=True
    )
    print("Direct SentenceTransformer.encode shape:", example_vector.shape)
    print("Model input-token limit:", dense_index.embedder.max_seq_length)


## 4. Inspect normalization and compute search manually

Each matrix row corresponds to one chunk. Encode a query into the same space, normalize
both sides, multiply the matrix by the query, then take the highest scores. Re-embedding
with a different model requires rebuilding the index; dimensions alone do not establish
compatibility.


In [ ]:
query = "How much time off can I take each year?"
vectors = np.asarray(dense_index.vectors)
query_vector = np.asarray(dense_index.embed_query(query)).reshape(-1)
unit_vectors = unit_rows(vectors)
unit_query = query_vector / max(np.linalg.norm(query_vector), 1e-12)
similarities = unit_vectors @ unit_query
best_indices = np.argsort(-similarities, kind="stable")[:3]
manual_hits = [
    {"id": chunks[i].id, "text": chunks[i].text, "metadata": chunks[i].metadata,
     "score": float(similarities[i])}
    for i in best_indices
]
display(show_hits(manual_hits))
print("Query dimensions:", query_vector.shape)
print("First five row norms:", np.linalg.norm(vectors, axis=1)[:5].round(4))


## 5. Compare lexical and semantic retrieval

Inspect the actual passages, not just score magnitudes. Scores from different retrieval
systems are not directly comparable. A paraphrase may help semantic search stand out;
an exact term such as `Retry-After` may favor lexical matching.
In offline mode both methods are lexical baselines, so no semantic improvement is claimed.


In [ ]:
comparison_rows = []
for name, index in [("TF-IDF", lexical_index), (dense_index.backend, dense_index)]:
    for rank, hit in enumerate(index.search(query, k=3), start=1):
        comparison_rows.append({"retriever": name, "rank": rank,
                                "source": hit["metadata"].get("source_id", hit["id"]),
                                "score": round(hit["score"], 4), "text": hit["text"]})
display(pd.DataFrame(comparison_rows))


## 6. Plot a query-to-document similarity matrix

The heatmap shows whether each query has a strong match in the expected source family.
A warm cell means similarity under the active backend, not proof of factual support.
Multiple chunks from a source can create several high cells.


In [ ]:
demo_queries = ["annual vacation allowance", "working from home permission",
                "too many API requests", "money back for an unused subscription"]
query_matrix = np.vstack([dense_index.embed_query(text) for text in demo_queries])
similarity_matrix = unit_rows(query_matrix) @ unit_vectors.T
labels = [chunk.metadata.get("source_id", chunk.id) for chunk in chunks]
fig, ax = plt.subplots(figsize=(max(10, len(chunks) * 0.38), 4.5))
heat = ax.imshow(similarity_matrix, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(labels)), labels, rotation=75, ha="right", fontsize=8)
ax.set_yticks(range(len(demo_queries)), demo_queries)
ax.set_title(f"Query × chunk cosine similarity — {dense_index.backend}")
fig.colorbar(heat, ax=ax, label="Cosine similarity")
fig.tight_layout()
plt.show()


## 7. Measure retrieval on a small labeled set

We score source IDs rather than chunk IDs because any chunk from the relevant source
counts for this introductory check. This can overestimate quality when the right source
has the wrong passage; advanced evaluation should label supporting passages too.
The four handcrafted queries are a demonstration set, not a statistically strong benchmark.


In [ ]:
labeled_queries = [
    ("How much time off can I take each year?", "benefits_leave"),
    ("Can I do my job from home several days each week?", "remote_work"),
    ("What happens if my application sends requests too quickly?", "api_limits"),
    ("Can I get my money back for a yearly plan I never used?", "refunds"),
]
evaluation_rows = []
for name, index in [("TF-IDF", lexical_index), (dense_index.backend, dense_index)]:
    for text, expected_source in labeled_queries:
        hits = index.search(text, k=3)
        sources = [hit["metadata"].get("source_id", hit["id"]) for hit in hits]
        evaluation_rows.append({"retriever": name, "query": text,
                                "expected": expected_source, "top_source": sources[0],
                                "hit@1": sources[0] == expected_source,
                                "hit@3": expected_source in sources})
evaluation = pd.DataFrame(evaluation_rows)
display(evaluation)
display(evaluation.groupby("retriever")[["hit@1", "hit@3"]].mean())


## 8. Connect retrieved vectors to Claude

The generator receives the selected source text, not the raw embedding coordinates.
Read the retrieved passages first and check the final claims and citations afterward.
This is the only Claude call in this notebook.


In [ ]:
final_question = "How many requests per minute does the Team plan allow, and what should I do after a 429?"
final_hits = dense_index.search(final_question, k=3)
display(show_hits(final_hits))
final_answer = rag_answer(final_question, final_hits, max_tokens=300)
display(Markdown(final_answer["text"]))
print("Verify evidence for: Team 600 requests/minute, HTTP 429, and Retry-After.")


## 9. Exercise: write reusable vector search

**Try first:** write a function that accepts a query and `k`, computes cosine scores, and
returns `(chunk_id, score)` pairs. Clamp `k` to the corpus length and reject `k <= 0`.
Try "professional development allowance" and inspect whether it finds the learning budget.
For a zero query vector, return no hits instead of arbitrary zero-score neighbors.


In [ ]:
# Sample solution. This directly uses the matrix, not DenseIndex.search().
def vector_search(text, k=3):
    if k <= 0:
        raise ValueError("k must be positive")
    vector = np.asarray(dense_index.embed_query(text)).reshape(-1)
    norm = np.linalg.norm(vector)
    if norm <= 1e-12:
        return []
    scores = unit_vectors @ (vector / norm)
    positions = np.argsort(-scores, kind="stable")[:min(k, len(chunks))]
    return [(chunks[position].id, float(scores[position])) for position in positions]

display(pd.DataFrame(vector_search("professional development allowance", k=3),
                     columns=["chunk_id", "cosine"]))


## Instructor variations and next step

Compare "vacation" with "paid leave", then compare exact technical terms with paraphrases.
Discuss why one successful synonym example does not establish overall quality. Increase
`k` and inspect additional context: can you identify irrelevant passages before calling Claude?

**Checkpoint:** explain what each matrix axis means, why normalized dot products implement
cosine, and why changing an embedding model requires re-indexing the documents.

Continue with notebook **04** for more advanced retrieval.

Official references: [Sentence Transformer API](https://www.sbert.net/docs/package_reference/sentence_transformer/model.html),
[all-MiniLM-L6-v2 model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
[scikit-learn TF-IDF](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html),
[NumPy matrix multiplication](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html).
